# Q10-Q11: CGLS Implementation and Small Bag Reconstruction

## Objectives
1. Test CGLS algorithm on toy problem
2. Verify convergence behavior
3. Reconstruct small bag with different λ values
4. Choose optimal regularization parameter

## Theory
CGLS solves: $\min_x \frac{1}{2}\|Ax - y\|^2 + \frac{\lambda}{2}\|Lx\|^2$

Advantages:
- Avoids computing $A^T A$ explicitly
- Only requires matrix-vector products with A and $A^T$
- $O(nnz(A))$ complexity per iteration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from src.data_utils import (
    load_toy_measurements, 
    load_3d_data,
    measurements_to_vector,
    vector_to_grid
)
from src.matrix_construction import (
    build_toy_ray_path_matrix,
    build_combined_derivative_matrix
)
from src.solvers import cgls
from src.visualization import (
    plot_convergence,
    visualize_2d_field,
    visualize_3d_slices
)

%matplotlib inline

## Q10: Test CGLS on Toy Problem

In [ ]:
# Load toy problem measurements
Y = load_toy_measurements('../data/Y.mat')
print(f"Measurement matrix shape: {Y.shape}")

# Convert to vector
y = measurements_to_vector(Y)
print(f"Measurement vector length: {len(y)}")

In [ ]:
# Build ray-path matrix A
M, N = 5, 5
A = build_toy_ray_path_matrix(M, N, delta_x=1.0, delta_y=1.0)
print(f"Ray-path matrix shape: {A.shape}")
print(f"Number of non-zeros: {np.count_nonzero(A)}")

In [ ]:
# Build derivative operator
L = build_combined_derivative_matrix(M, N, dimensions=2)
print(f"Derivative matrix shape: {L.shape}")

In [ ]:
# Solve with CGLS for different λ values
lambda_values = [1e-6, 1e-5, 1e-4, 1e-3]
results = {}

for lam in lambda_values:
    print(f"\nSolving with λ = {lam}")
    result = cgls(A, y, L, lam=lam, tol=1e-6, max_iter=200)
    results[lam] = result
    print(f"  Converged: {result.converged}")
    print(f"  Iterations: {result.iterations}")
    print(f"  Final objective: {result.objective_values[-1]:.6e}")

In [ ]:
# Plot convergence for one λ value
lam = 1e-5
result = results[lam]
plot_convergence(result.residuals, result.objective_values, 
                title=f"CGLS Convergence (λ={lam})",
                save_path=f'../results/figures/cgls_toy_convergence_lam{lam}.png')

In [ ]:
# Visualize reconstructions
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

for idx, lam in enumerate(lambda_values):
    X_recon = vector_to_grid(results[lam].x, (M, N))
    im = axes[idx].imshow(X_recon, cmap='gray', origin='lower')
    axes[idx].set_title(f'λ = {lam}')
    plt.colorbar(im, ax=axes[idx])

plt.tight_layout()
plt.savefig('../results/figures/toy_reconstructions_comparison.png', dpi=150)
plt.show()

## Q11: Reconstruct Small Bag

In [ ]:
# Load 3D data
y_small, A_small = load_3d_data('../data/Small')
n = 19  # Volume size for small bag
print(f"Volume size: {n}×{n}×{n} = {n**3}")

In [ ]:
# Build 3D derivative operator
L_3d = build_combined_derivative_matrix(n, n, dimensions=3)
print(f"3D derivative matrix shape: {L_3d.shape}")

In [ ]:
# Test different regularization parameters
lambda_values_3d = [1e-6, 1e-5, 1e-4, 1e-3]
results_3d = {}

for lam in lambda_values_3d:
    print(f"\nSolving 3D problem with λ = {lam}...")
    result = cgls(A_small, y_small, L_3d, lam=lam, tol=1e-6, max_iter=300)
    results_3d[lam] = result
    
    print(f"  Converged: {result.converged}")
    print(f"  Iterations: {result.iterations}")
    print(f"  Final objective: {result.objective_values[-1]:.6e}")
    
    # Convert to 3D grid
    X_recon = vector_to_grid(result.x, (n, n, n))
    
    # Visualize middle slices
    visualize_3d_slices(X_recon, axis='z', num_slices=9,
                       title=f"Small Bag Reconstruction (λ={lam})",
                       save_path=f'../results/figures/small_bag_lam{lam}.png')

## Comparison Table

In [ ]:
# Create comparison table
import pandas as pd

comparison_data = []
for lam in lambda_values_3d:
    result = results_3d[lam]
    comparison_data.append({
        'λ': lam,
        'Iterations': result.iterations,
        'Final Objective': f"{result.objective_values[-1]:.6e}",
        'Converged': 'Yes' if result.converged else 'No'
    })

df = pd.DataFrame(comparison_data)
print("\nComparison of Regularization Parameters:")
print(df.to_string(index=False))

## Analysis: Choose Best λ

**Your answer here:**

Based on the visual quality and convergence behavior, I choose λ = _____ because:
- 
- 
- 

Trade-offs:
- Too small λ: overfitting, noisy reconstruction
- Too large λ: over-smoothing, loss of detail


In [ ]:
# Plot convergence for best λ
best_lam = 1e-5  # Change this based on your choice
best_result = results_3d[best_lam]

plot_convergence(best_result.residuals, best_result.objective_values,
                title=f"CGLS Convergence for Small Bag (λ={best_lam})",
                save_path='../results/figures/small_bag_best_convergence.png')